In [1]:
# Daily challenge GOLD: DNA

import random

class Gene:
    """
    Represents a single gene with a binary value (0 or 1).
    It can mutate (flip its value).
    """
    def __init__(self, value=None):
        """
        Initializes a Gene. If no value is provided, it's randomly set to 0 or 1.
        """
        if value is None:
            self.value = random.randint(0, 1)
        elif value in [0, 1]:
            self.value = value
        else:
            raise ValueError("Gene value must be 0 or 1.")

    def mutate(self):
        """
        Flips the gene's value (0 becomes 1, 1 becomes 0).
        """
        self.value = 1 - self.value

    def __repr__(self):
        """String representation for debugging."""
        return str(self.value)

class Chromosome:
    """
    A series of 10 Genes. It can mutate, causing a random number of its genes to flip.
    """
    CHROMOSOME_LENGTH = 10

    def __init__(self, initial_genes=None):
        """
        Initializes a Chromosome with 10 Genes.
        If initial_genes is None, genes are randomly generated.
        Otherwise, it takes a list of Gene objects or 0/1 values.
        """
        if initial_genes:
            if len(initial_genes) != self.CHROMOSOME_LENGTH:
                raise ValueError(f"Chromosome must have {self.CHROMOSOME_LENGTH} genes.")
            self.genes = [Gene(g) if not isinstance(g, Gene) else g for g in initial_genes]
        else:
            self.genes = [Gene() for _ in range(self.CHROMOSOME_LENGTH)]

    def mutate(self):
        """
        Mutates the chromosome: a random number of genes (from 0 to CHROMOSOME_LENGTH)
        are randomly selected and flipped. Each selected gene has a 1/2 chance to flip.
        """
        num_genes_to_consider = random.randint(0, self.CHROMOSOME_LENGTH)
        genes_to_potentially_flip = random.sample(self.genes, num_genes_to_consider)

        for gene in genes_to_potentially_flip:
            if random.random() < 0.5: # 1/2 chance to flip
                gene.mutate()

    def get_sequence(self):
        """Returns the sequence of gene values as a list of integers."""
        return [gene.value for gene in self.genes]

    def is_all_ones(self):
        """Checks if all genes in the chromosome are 1s."""
        return all(gene.value == 1 for gene in self.genes)

    def __repr__(self):
        """String representation for debugging."""
        return f"[{''.join(str(g) for g in self.genes)}]"

class DNA:
    """
    A series of 10 Chromosomes. It can mutate, causing a random number of its chromosomes to mutate.
    """
    DNA_LENGTH = 10

    def __init__(self, initial_chromosomes=None):
        """
        Initializes DNA with 10 Chromosomes.
        If initial_chromosomes is None, chromosomes are randomly generated.
        Otherwise, it takes a list of Chromosome objects.
        """
        if initial_chromosomes:
            if len(initial_chromosomes) != self.DNA_LENGTH:
                raise ValueError(f"DNA must have {self.DNA_LENGTH} chromosomes.")
            self.chromosomes = initial_chromosomes
        else:
            # Create chromosomes with randomly initialized genes (default Chromosome behavior)
            self.chromosomes = [Chromosome() for _ in range(self.DNA_LENGTH)]

    def mutate(self):
        """
        Mutates the DNA: a random number of chromosomes (from 0 to DNA_LENGTH)
        are randomly selected and mutated (their own mutate method is called).
        Each selected chromosome has a 1/2 chance to mutate.
        """
        num_chromosomes_to_consider = random.randint(0, self.DNA_LENGTH)
        chromosomes_to_potentially_mutate = random.sample(self.chromosomes, num_chromosomes_to_consider)

        for chromosome in chromosomes_to_potentially_mutate:
            if random.random() < 0.5: # 1/2 chance to mutate
                chromosome.mutate()

    def is_all_ones(self):
        """Checks if all genes in all chromosomes of the DNA are 1s."""
        return all(chromosome.is_all_ones() for chromosome in self.chromosomes)

    def __repr__(self):
        """String representation for debugging."""
        return "\n".join(str(c) for c in self.chromosomes)

class Organism:
    """
    Represents an organism with a DNA object and an environment parameter
    that sets the probability for its DNA to mutate each generation.
    """
    def __init__(self, dna_object, environment_mutation_prob=0.1):
        """
        Initializes an Organism.

        Args:
            dna_object (DNA): The DNA object for this organism.
            environment_mutation_prob (float): Probability (0.0 to 1.0)
                                               that the DNA will attempt to mutate
                                               in a given generation.
        """
        if not isinstance(dna_object, DNA):
            raise TypeError("dna_object must be an instance of the DNA class.")
        if not (0.0 <= environment_mutation_prob <= 1.0):
            raise ValueError("environment_mutation_prob must be between 0.0 and 1.0.")

        self.dna = dna_object
        self.environment_mutation_prob = environment_mutation_prob

    def live_a_generation(self):
        """
        Simulates one generation for the organism.
        Based on environment_mutation_prob, its DNA might mutate.
        """
        if random.random() < self.environment_mutation_prob:
            self.dna.mutate()

    def get_dna(self):
        """Returns the DNA object of the organism."""
        return self.dna

    def __repr__(self):
        return f"Organism(DNA_state={self.dna.is_all_ones()}, Env_Prob={self.environment_mutation_prob})"

# --- Main Simulation ---
def run_simulation(num_organisms=5, environment_prob=0.5, max_generations=10000):
    """
    Instantiates a number of Organisms and lets them mutate until one gets
    to a DNA which is only made of 1s. Records the number of generations.

    Args:
        num_organisms (int): The number of organisms to simulate.
        environment_prob (float): The mutation probability for each organism's DNA per generation.
        max_generations (int): Maximum generations to run the simulation.

    Returns:
        tuple: (winning_organism_index, generations_taken) or (None, max_generations) if no winner.
    """
    print("--- Biology Research: DNA Mutation Simulation ---")
    print(f"Simulating {num_organisms} organisms, each with a DNA of 10 chromosomes (10 genes each).")
    print(f"Environment mutation probability per organism per generation: {environment_prob}")
    print(f"Maximum generations: {max_generations}\n")

    organisms = []
    for i in range(num_organisms):
        # Initialize DNA for each organism, starting with random genes (0 or 1)
        initial_dna = DNA()
        organism = Organism(initial_dna, environment_mutation_prob=environment_prob)
        organisms.append(organism)
        print(f"Organism {i+1} created. Initial DNA state (all ones?): {organism.get_dna().is_all_ones()}")

    generations_taken = 0
    winner_index = -1

    for gen in range(max_generations):
        generations_taken = gen + 1
        found_winner = False
        for i, organism in enumerate(organisms):
            organism.live_a_generation() # Simulate one generation
            if organism.get_dna().is_all_ones():
                winner_index = i
                found_winner = True
                break # Found a winner, stop this generation's loop

        if found_winner:
            print(f"\n--- Simulation Complete ---")
            print(f"Organism {winner_index + 1} achieved all 1s DNA!")
            print(f"It took {generations_taken} generations.")
            print("\nFinal DNA of the winning organism:")
            print(organisms[winner_index].get_dna())
            return winner_index, generations_taken

        if (gen + 1) % (max_generations // 10) == 0: # Print progress update
            print(f"Generation {gen + 1}/{max_generations} reached. No winner yet.")

    print(f"\n--- Simulation Ended ---")
    print(f"Max generations ({max_generations}) reached. No organism achieved all 1s DNA.")
    return None, max_generations

# --- Run the Simulation ---
if __name__ == "__main__":
    # You can adjust these parameters to see how they affect the outcome
    # Higher environment_prob means more mutations, potentially faster convergence
    # but also potentially overshooting the target if mutations are too frequent.
    # More organisms increase the chance of one hitting the target.
    winning_organism_idx, generations = run_simulation(
        num_organisms=5,
        environment_prob=0.1, # Probability of DNA mutation per organism per generation
        max_generations=50000 # Set a reasonable limit to avoid infinite loops
    )

    print("\n--- Biology Research Notebook Entry ---")
    if winning_organism_idx is not None:
        print(f"Conclusion: An organism successfully evolved to have all '1's in its DNA.")
        print(f"It took {generations} generations for Organism {winning_organism_idx + 1} to reach this state.")
        print(f"This demonstrates how random mutations, even with a low probability,")
        print(f"can lead to a specific genetic configuration over generations.")
    else:
        print(f"Conclusion: No organism achieved all '1's in its DNA within {generations} generations.")
        print(f"This could be due to a low environment mutation probability, too few organisms,")
        print(f"or simply the stochastic nature of random mutations over the set maximum generations.")
        print(f"Consider increasing max_generations, num_organisms, or environment_prob for a higher chance of success.")



--- Biology Research: DNA Mutation Simulation ---
Simulating 5 organisms, each with a DNA of 10 chromosomes (10 genes each).
Environment mutation probability per organism per generation: 0.1
Maximum generations: 50000

Organism 1 created. Initial DNA state (all ones?): False
Organism 2 created. Initial DNA state (all ones?): False
Organism 3 created. Initial DNA state (all ones?): False
Organism 4 created. Initial DNA state (all ones?): False
Organism 5 created. Initial DNA state (all ones?): False
Generation 5000/50000 reached. No winner yet.
Generation 10000/50000 reached. No winner yet.
Generation 15000/50000 reached. No winner yet.
Generation 20000/50000 reached. No winner yet.
Generation 25000/50000 reached. No winner yet.
Generation 30000/50000 reached. No winner yet.
Generation 35000/50000 reached. No winner yet.
Generation 40000/50000 reached. No winner yet.
Generation 45000/50000 reached. No winner yet.
Generation 50000/50000 reached. No winner yet.

--- Simulation Ended ---
M